# Stage 05b: Hyperparameter Optimization

**Purpose:** Find optimal XGBoost hyperparameters using actuarial lift metrics

**Inputs:**
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet
- data/04b_train.parquet (for target/exposure)
- data/04b_test.parquet
- config_generated/04c_monotonicity_constraints.yaml

**Outputs:**
- results/05b_best_params.yaml (HPO-tuned parameters only)
- results/05b_hpo_results.csv (all trial results)
- results/05b_metrics.yaml (best fit_quality, model_power, score)

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import xgboost as xgb
import yaml, os, sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error
from itertools import product

sys.path.insert(0, str(Path.cwd() / "lib"))
from utils import setup_notebook_environment
from hpo_metrics import calculate_model_metrics, calculate_lift_opt_score

print("#" * 40)
print("# STAGE 05b: HYPERPARAMETER OPTIMIZATION")
print("#" * 40)

project_root = setup_notebook_environment()
from data_preparation import prepare_model_data


In [ ]:
print(f"Python: {sys.version}")
print(f"XGBoost: {xgb.__version__}")

In [ ]:
# Get machine config
pc_num = open("current.pc").read().strip()
pc_id = f"PC{pc_num}"

config_file = f"{config_path}/config.yaml"
with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

output_base = cfg["machines"][pc_id]["paths"]["output_path"]
target = cfg["experiment"]["target"]
exposure = cfg["experiment"]["exposure"]
join_key = cfg["data"]["join_key"]

print(f"\nOutput: {output_base}")
print(f"Target: {target}")
print(f"Exposure: {exposure}")
target_method = cfg["model"].get("target_method", "direct")
print(f"Target method: {target_method}")

In [ ]:
# Prepare data with loss cap filtering
loss_cap = cfg['experiment'].get('loss_cap', None)
loss_type = cfg['experiment'].get('loss_type', 'bi')
exposure_floor = cfg['experiment'].get('exposure_floor', None)

data = prepare_model_data(
    output_base=output_base,
    target_col=target,
    exposure_col=exposure,
    target_method=target_method,
    loss_cap=loss_cap,
    
    loss_type=loss_type,
    exposure_floor=exposure_floor,
    verbose=True
)

# Extract prepared data
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
w_train = data['w_train']
w_test = data['w_test']
pp_train_actual = data['pp_train']
pp_test_actual = data['pp_test']
glm_train = data['glm_train']
glm_test = data['glm_test']
train_orig = data['train_orig']
test_orig = data['test_orig']
top_100_report = data['top_100_report']

# Save top 100 report if generated
if top_100_report is not None:
    report_path = f"{output_base}/reports"
    import os
    os.makedirs(report_path, exist_ok=True)
    top_100_report.to_csv(f"{report_path}/top_100_targets_before_cap.csv", index=True)
    print(f"  Saved top 100 report to: {report_path}/top_100_targets_before_cap.csv")

print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")
print(f"  Features: {X_train.shape[1]}")

In [ ]:
# Load monotonicity constraints
mono_file = f"{output_base}/config_generated/04c_monotonicity_constraints.yaml"
if os.path.exists(mono_file):
    print(f"\n* Loading monotonicity constraints...")
    with open(mono_file, "r") as f:
        mono_dict = yaml.safe_load(f)
    
    # Build constraints tuple (default to 0 if feature not in dict)
    monotone_constraints = tuple(mono_dict.get(col, 0) for col in X_train.columns)
    
    # Check for missing features
    missing = [col for col in X_train.columns if col not in mono_dict]
    if missing:
        print(f"  Warning: {len(missing)} features not in constraints (defaulting to 0)")
        print(f"  First few: {missing[:5]}")
    
    print(f"  Loaded {len(monotone_constraints)} constraints")
else:
    monotone_constraints = None
    print(f"\n* No monotonicity constraints found")

In [ ]:
# Load HPO configuration
method = cfg.get('hyperparameter_optimization', {}).get('method', 'grid')
param_grid = cfg.get('hyperparameter_optimization', {}).get('param_grid')
scoring_config = cfg.get('hyperparameter_optimization', {}).get('scoring', {})

print(f"\n* HPO Method: {method}")
print(f"  Scoring config: {scoring_config}")

# Prepare base XGBoost params
base_xgb_params = cfg['xgboost'].copy()
base_xgb_params.pop('n_estimators', None)

# Fix eval_metric for tweedie
if 'eval_metric' in base_xgb_params and 'tweedie' in base_xgb_params['eval_metric']:
    if '@' not in base_xgb_params['eval_metric']:
        variance_power = base_xgb_params.get('tweedie_variance_power', 1.5)
        base_xgb_params['eval_metric'] = f"{base_xgb_params['eval_metric']}@{variance_power}"

if method == 'optuna':
    # Optuna optimization
    from hpo_optuna import run_optuna_search
    n_trials = cfg.get('hyperparameter_optimization', {}).get('n_trials', 100)
    
    print(f"\n* Running Optuna Search ({n_trials} trials)...")
    hpo_results = run_optuna_search(
        X_train, y_train, w_train,
        X_test, y_test, w_test,
        param_grid, base_xgb_params,
        monotone_constraints, exposure,
        scoring_config,
        n_trials=n_trials,
        glm_pred_test=glm_test,
        target_method=target_method
    )
elif method == 'grid':
    # Grid search
    from hpo_runner import run_grid_search
    
    # Verify n_estimators is in param_grid
    if 'n_estimators' not in param_grid:
        raise ValueError("n_estimators must be in param_grid for HPO!")
    
    print(f"\n* Running Grid Search...")
    hpo_results = run_grid_search(
        X_train, y_train, w_train,
        X_test, y_test, w_test,
        param_grid, base_xgb_params,
        monotone_constraints, exposure,
        scoring_config,
        glm_pred_test=glm_test,
        target_method=target_method
    )
else:
    raise ValueError(f"Unknown HPO method: {method}. Use 'grid' or 'optuna'")

best_params = hpo_results['best_params']
best_metrics = hpo_results['best_metrics']
results_df = hpo_results['results_df']


In [ ]:
# Save HPO results
from hpo_runner import save_hpo_results

save_hpo_results(output_base, best_params, best_metrics, results_df)